In [28]:
import openai
from pinecone import Pinecone, ServerlessSpec          # v3 client
from langchain_pinecone import PineconeVectorStore     # new LC wrapper
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.docstore.document import Document
from dotenv import load_dotenv
import os
from conversation_manager import ConversationManager

load_dotenv()
# set API key
OPENAI_KEY=os.getenv("OPENAI_API_KEY")
OPENAI_BASE=os.getenv("OPENAI_API_BASE")
PINECONE_KEY = os.getenv('PINECONE_API_KEY')

# Debug: Check if environment variables are loaded
print(f"OPENAI_KEY loaded: {'Yes' if OPENAI_KEY else 'No'}")
print(f"OPENAI_BASE loaded: {'Yes' if OPENAI_BASE else 'No'}")
print(f"PINECONE_KEY loaded: {'Yes' if PINECONE_KEY else 'No'}")

# Check if any required keys are missing
if not OPENAI_KEY:
    print("ERROR: OPENAI_API_KEY is not set!")
if not PINECONE_KEY:
    print("ERROR: PINECONE_API_KEY is not set!")


openai_client = openai.OpenAI(
    base_url = OPENAI_BASE,
    api_key=OPENAI_KEY)


OPENAI_KEY loaded: Yes
OPENAI_BASE loaded: Yes
PINECONE_KEY loaded: Yes


In [24]:
# get the history data 
from google.cloud import bigquery
client=bigquery.Client(project='khanacademy.org:deductive-jet-827')
query = """
with lib as (
SELECT DISTINCT
      content_path.content_slug,
      content_path.content_id
    FROM `khan-core.content.published_content_paths_daily`
    WHERE dt = '2025-10-06'
      AND locale = 'en'
      and content_path.domain_slug='math'
      
      )
, threads as ( 
select distinct a.userKaid as kaid,
 a.aiGuideThreadId, 
 a.contentId,
 a.contentKind, 
 a.delta_dt,
 b.content_slug
 FROM `khan-data-lake.analytics_events.AiGuideInteractionStored_materialized` a
left join lib b
 
on a.contentId=b.content_id 
where a.eventInfo.eventTime>'2025-08-05'
and a.aiGuideExperienceReportingLabelId='content-tutoring'
and a.userKaid='kaid_248328473686127753888179'
)
, text as (
select kaid, 
threadId, 
string_agg(concat(conversationEntryType, ": ", text), " ") as thread_text
from (
    select kaid,
    threadId, 
    conversationEntryType, text
    from `khan-iris.aiguide.conversation_entries`
    where dt > '2025-08-05'
    and threadID in (select aiGuideThreadId from threads)
    --AND threadID in (select aiGuideThreadId from user_lengths)
    order by threadID, conversationEntryTime
) group by 1, 2  
)
-- need tp adjust fpm to just get current state at today
, fpm AS (
  SELECT distinct
    a.kaid,
    a.skill_id,
  -- a.skill_fpm_before_level,
  --  a.skill_fpm_after_level
    LAST_VALUE(a.skill_fpm_after_level) OVER (PARTITION BY a.kaid, skill_id ORDER BY a.fpm_event_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_fpm_level
    FROM `khan-core.mastery.fpm_events` as a
    where fpm_event_dt between '2025-08-05' and CURRENT_DATE()
    and kaid in (select kaid from threads)
  and task_type= 'practice_tutorial'
)
select
  kaid, 
  aiGuideThreadId as turn_id,
  content_slug as skill_tag,
  delta_dt as dt,
  last_fpm_level as fpm_state,
  thread_text as text
  from (
select threads.*, 
  text.thread_text,
  --fpm.skill_fpm_before_level,
  fpm.last_fpm_level  
  from threads 
  left join text on 
  threads.kaid=text.kaid and 
  threads.AiGuideThreadId = text.threadId
  left join fpm on 
  threads.kaid=fpm.kaid and 
  threads.contentID = fpm.skill_id
  ) 
 """

# query with prereqs and fpm on those
query2="""
with lib as (
SELECT DISTINCT
      content_path.content_slug,
      content_path.content_id
    FROM `khan-core.content.published_content_paths_daily`
    WHERE dt = '2025-10-06'
      AND locale = 'en'
      and content_path.domain_slug='math'
      
      )
, threads as ( 
select distinct a.userKaid as kaid,
 a.aiGuideThreadId, 
 a.contentId,
 a.contentKind, 
 a.delta_dt,
 b.content_slug
 FROM `khan-data-lake.analytics_events.AiGuideInteractionStored_materialized` a
left join lib b
 
on a.contentId=b.content_id 
where a.eventInfo.eventTime>'2025-08-05'
and a.aiGuideExperienceReportingLabelId='content-tutoring'
and a.userKaid='kaid_248328473686127753888179'
)
, text as (
select kaid, 
threadId, 
string_agg(concat(conversationEntryType, ": ", text), " ") as thread_text
from (
    select kaid,
    threadId, 
    conversationEntryType, text
    from `khan-iris.aiguide.conversation_entries`
    where dt > '2025-08-05'
    and threadID in (select aiGuideThreadId from threads)
    --AND threadID in (select aiGuideThreadId from user_lengths)
    order by threadID, conversationEntryTime
) group by 1, 2  
)
-- need tp adjust fpm to just get current state at today
, fpm AS (
  SELECT distinct
    a.kaid,
    a.skill_id,
  -- a.skill_fpm_before_level,
  --  a.skill_fpm_after_level
    LAST_VALUE(a.skill_fpm_after_level) OVER (PARTITION BY a.kaid, skill_id ORDER BY a.fpm_event_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS last_fpm_level
    FROM `khan-core.mastery.fpm_events` as a
    where fpm_event_dt between '2025-08-05' and CURRENT_DATE()
    --and kaid='kaid_248328473686127753888179'
    and kaid in (select kaid from threads)
  and task_type= 'practice_tutorial'
) 
, prereqs as (select a.*,
 lib.content_id as prereq_content_id from (
 SELECT
  learnableContentRevision.slug,
  learnableContentRevision.contentId,
  json_value(prereq_element) AS prerequisite,
  MAX(eventInfo.eventTime) AS latest_version,
FROM
  `khan-data-lake.analytics_events.ExerciseRevisionPublish_materialized`  ,
  UNNEST(JSON_EXTRACT_ARRAY(prerequisitesJson)) AS prereq_element
WHERE
  learnableContentRevision.kaLocale = 'en'
GROUP BY
  learnableContentRevision.slug,
  learnableContentRevision.contentId,
  prereq_element
) a
  left join lib on 
  lib.content_slug = a.prerequisite
  ) 
, prereq_fpm as (
  SELECT distinct
    a.kaid,
    a.skill_id as prereq_content_id,
  -- a.skill_fpm_before_level,
  --  a.skill_fpm_after_level
    LAST_VALUE(a.skill_fpm_after_level) OVER (PARTITION BY a.kaid, skill_id ORDER BY a.fpm_event_ts ASC ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS prereq_fpm_level
    FROM `khan-core.mastery.fpm_events` as a
    where fpm_event_dt between '2024-08-05' and CURRENT_DATE()
    --and kaid='kaid_248328473686127753888179'
    and kaid in (select kaid from threads)
    and skill_id in (select prereq_content_id from prereqs)
    and task_type= 'practice_tutorial'
)
, hist as (
select
  kaid, 
  aiGuideThreadId as turn_id,
  contentId,
  content_slug as skill_tag,
  delta_dt as dt,
  last_fpm_level as fpm_state,
  thread_text as text
  from (
select threads.*, 
  text.thread_text,
  --fpm.skill_fpm_before_level,
  fpm.last_fpm_level
  from threads 
  left join text on 
  threads.kaid=text.kaid and 
  threads.AiGuideThreadId = text.threadId
  left join fpm on 
  threads.kaid=fpm.kaid and 
  threads.contentID = fpm.skill_id
  )
)
select 
  kaid, 
  turn_id,
  skill_tag,
  dt, 
  fpm_state,
  prerequisite,
  case when prereq_fpm_level is null then 'unknown' else prereq_fpm_level end as prereq_fpm_level,
  text
  from (
select hist.*,
b.prereq_content_id,
b.prerequisite ,
c.prereq_fpm_level
from hist 
left join prereqs b 
on hist.contentId = b.contentId
left join prereq_fpm c 
on b.prereq_content_id = c.prereq_content_id
)
"""

In [ ]:
# create vector DB and upsert docs
INDEX_NAME       = "student"
DIMENSION        = 1536                    
METRIC           = "cosine"                

# init pinecone 
pc = Pinecone(api_key=PINECONE_KEY)
# create index  
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(cloud="gcp", region="us-central1")
    )

index = pc.Index(INDEX_NAME)

# # Initialize the embedder
embedder = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_KEY
)
# create the vector store 
vector_store = PineconeVectorStore(index, embedder)

# build and upload docs 
def build_docs(rows):
    docs=[]
    for kaid, turn_id, skill_tag, dt, fpm_state, prerequisite, prereq_fpm_level, text in rows:
        metadata={'kaid': kaid, 
                  'turn_id': turn_id, 
                  'skill_tag': skill_tag, 
                  'date': str(dt), 
                  'fpm_state': fpm_state,
                  'prerequisite': prerequisite,
                  'prereq_fpm_level': prereq_fpm_level}
        docs.append(Document(page_content=text, metadata=metadata))
    return docs

rows=client.query(query2)
docs = build_docs(rows)

vector_store.add_documents(docs)

        

In [50]:
# create the memory block  (try "multiplication_3", or "regrouping-whole-numbers" )
# multiply-with-partial-products--2-digit-numbers-  # -- good example of familiar prereq
def build_memory_block(student_msg, kaid, skill_tag, k=4 ):

  vec_results = vector_store.similarity_search(student_msg, 
                                               k=k, 
                                               filter=({"kaid" : kaid, "skill_tag": skill_tag})
                                               )
  memory_chunks = []
  skill_state = {}
  prereq_state ={} 
  
  for doc in vec_results:
      m = doc.metadata
      # get past text 
      turns   = m.get("turn_id")
      skills  = m.get("skill_tag", [])
      dt = m.get("date", "")

      header = f"[turn {turns} | skills: {skills} | {dt}]"
      chunk  = f"{header}\n{doc.page_content}"
      memory_chunks.append(chunk)

      # get skill levels for current skill 
      skill  = m.get("skill_tag", [])
      skill_level = m.get("fpm_state", "")
      skill_state[skill] = skill_level
  
      # get prereq levels 
      prereq = m.get("prerequisite", "")
      prereq_level = m.get("prereq_fpm_level", "unknown")
      prereq_state[prereq] = prereq_level

  memory_block = "\n\n".join(memory_chunks)          
  
  skill_lines = [f"{skill}: {level}" for skill, level in skill_state.items()]
  skill_block = "\n".join(skill_lines)  
  
  # Filter out empty prerequisite keys and format properly
  prereq_lines = [f"{prereq}: {level}" for prereq, level in prereq_state.items() if prereq.strip()]
  prereq_block = "; ".join(prereq_lines).strip()  # Remove leading/trailing whitespace
     
  return memory_block, skill_block, prereq_block
  

In [55]:
# Student persona system prompt
student_system_prompt = ("""
You are a student who is trying to solve a math problem. However, you are generally disengaged 
and you have gaps in math knowledge. You only seek answers and solutions from a tutor and avoid doing own thinking.
Do not say "as a student I would ...", instead respond as a student who is just fishing or answers and only 
answers in one or two word sentences.
""")

# Tutor system prompt
tutor_system_prompt = ("""
# ROLE 
You are an AI math tutor.

# YOUR PEDAGOGY RULES
1. Use Socratic questioning.
2. Encourage student to explain their reasoning aloud.
3. Do not do the work for the student and do not give away answers. 

# CRITICAL: PREREQUISITE SKILL INSTRUCTIONS
You will receive specific data about:
- Current skill knowledge state
- Prerequisite skill states  

**IMPORTANT**: You must ONLY use the prerequisite skills that are explicitly provided to you in the "Prerequisite skill states" section. DO NOT assume or mention any prerequisite skills that are not listed there.

Skill and prerequisite skill levels can be "unknown", "attempted", "familiar", "proficient" or "mastered".
Proficient and mastered are the highest levels of skill mastery. Familair, attempted, and unknown levels mean
that a student does not know that material. 

# TUTORING APPROACH BASED ON PROVIDED DATA:
1. **Check the provided prerequisite skills ONLY**:
   - If ANY prerequisite skill listed is at "familiar" level or lower, suggest reviewing that SPECIFIC skill first
   - Use the exact skill name as provided in the prerequisite data
   - Offer an example problem as a warm-up for that specific prerequisite skill. 
   - If a student solves that warm-up problem correctly, proceed to the current skill. 
   - If a student solves that warm-up problem incorrectly, offer a link to a video on the prerequisite skill. 
   
2. **If all provided prerequisite skills are proficient/mastered**:
   - Look at the current skill level. 
   
3. **If no prerequisite skills are provided**:
   - Proceed directly with the current skill
""")

# Legacy student function (kept for backward compatibility)
def student(student_system_prompt, tutor_message, model="gpt-4-khan"):
    """
    Legacy student function - generates student response based on tutor message.
    NOTE: This is kept for backward compatibility. New code should use ConversationManager.
    """
    completion = openai_client.chat.completions.create(
        model=model,
        temperature=0.0,
        messages=[{"role": "user", "content": student_system_prompt + tutor_message}])
    return completion.choices[0].message.content

# Legacy tutor function (kept for backward compatibility)  
def tutor_legacy(student_msg, kaid, skill_tag, model="gpt-4-khan", k=4, temperature=0.0):
    """
    Legacy tutor function - single response without conversation history.
    NOTE: This is kept for backward compatibility. New code should use ConversationManager.
    """
    memory_block, skill_set, prereq_set = build_memory_block(student_msg, kaid, skill_tag, k=k)
    
    completion = openai_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": tutor_system_prompt},
            {"role": "system", "content": 'Current knowledge state on the relevant skill: ' + skill_set},
            {"role": "system", "content": 'Relevant past conversations: ' + memory_block},
            {"role": "system", "content": 'Prerequisite skill states: ' + prereq_set},
            {"role": "user", "content": student_msg}
        ]) 
    return completion.choices[0].message.content
    


In [57]:
# Conversation simulation using ConversationManager
import pandas as pd
pd.set_option('display.max_colwidth', None)  # or a large number like 1000

# Initialize conversation manager
conversation_manager = ConversationManager(openai_client, build_memory_block)

def tutor(student_msg, kaid, skill_tag, model="gpt-4-khan", k=4, temperature=0.0, max_turns=5):
    """
    Run a tutoring conversation using the ConversationManager.
    
    Args:
        student_msg: Initial student message
        kaid: Student ID
        skill_tag: Skill being tutored
        model: OpenAI model to use
        k: Number of similar conversations to retrieve from memory
        temperature: Temperature setting for responses
        max_turns: Number of conversation turns (default 3 for backward compatibility)
        
    Returns:
        DataFrame with conversation questions and responses
    """
    return conversation_manager.run_conversation(
        student_msg=student_msg,
        kaid=kaid,
        skill_tag=skill_tag,
        tutor_system_prompt=tutor_system_prompt,
        student_system_prompt=student_system_prompt,
        max_turns=max_turns,
        k=k,
        model=model,
        temperature=temperature
    )

# Example usage - test the conversation system
student_msg = 'I am having trouble using partial products to multiply 85 times 61'
result = tutor(student_msg, 'kaid_248328473686127753888179', 'multiply-with-partial-products--2-digit-numbers-', k=4)
print("Conversation Results:")
result

Conversation Results:


,question,responses
0,I am having trouble using partial products to multiply 85 times 61,"I see that you're familiar with the skill of multiplying using partial products. Let's review this skill with a simpler problem first. \n\nCan you try to multiply 23 by 4 using partial products? Remember, you can break 23 into 20 and 3, and then multiply each part by 4."
1,No.,"That's okay! Let's break it down together. \n\nWhen we say 23, we actually mean 20 + 3. So, when we multiply 23 by 4, we can multiply each part by 4 separately. \n\nSo, what is 20 times 4? And what is 3 times 4?"
2,"80, 12.","Great job! Now, to find the product of 23 and 4 using partial products, we add these two results together. Can you do that?"
3,No.,"That's okay! Let's do it together. \n\nYou've found that 20 times 4 is 80 and 3 times 4 is 12. \n\nSo, if we add these two results together, we get 80 + 12. \n\nCan you try adding these two numbers together?"
4,92,"Excellent! So, 23 times 4 equals 92 when we use partial products. \n\nNow that we've reviewed this, let's try applying the same method to the problem you're working on: 85 times 61. \n\nRemember, you can break 85 into 80 and 5, and 61 into 60 and 1. Then, multiply each part separately. \n\nCan you try that?"


In [ ]:
# Advanced Usage Examples and Testing

# Example 1: Different conversation lengths
print("=== Testing different conversation lengths ===")
print("Short conversation (2 turns):")
short_result = tutor(
    student_msg='I need help with 23 * 47', 
    kaid='kaid_248328473686127753888179', 
    skill_tag=' multiply-with-partial-products--2-digit-numbers-', 
    max_turns=2
)
display(short_result)

print("\nLonger conversation (5 turns):")
long_result = tutor(
    student_msg='I am confused about how to break down multiplication problems', 
    kaid='kaid_248328473686127753888179', 
    skill_tag=' multiply-with-partial-products--2-digit-numbers-', 
    max_turns=5
)
display(long_result)

# Example 2: Access conversation history and memory context
print("\n=== Accessing conversation internals ===")
print("Conversation history:")
for i, msg in enumerate(conversation_manager.get_conversation_history()[-4:]):  # Show last 4 messages
    print(f"{i+1}. {msg['role']}: {msg['content'][:100]}...")

print("\nMemory context (static per session):")
memory, skills, prereqs = conversation_manager.get_memory_context()
print(f"Skills: {skills[:200]}...")
print(f"Prerequisites: {prereqs[:200]}...")

# Example 3: Testing with different student personas or skill tags
print("\n=== Testing flexibility ===")
print("This system now supports:")
print("- Configurable conversation lengths (max_turns parameter)")
print("- Proper conversation history maintenance")
print("- Static memory retrieval (once per session)")
print("- Access to conversation internals for debugging/analysis")
print("- Backward compatibility with existing functions")
